# 1. Business understanding  

The goal of this project is to create a model that predicts whether a person has diabetes or prediabetes based on answers to a health questionnaire. The dataset contains information about health conditions, lifestyle, and demographic factors.

The objective is to compare Logistic Regression and Random Forest models and determine which approach performs best for this specific dataset. Different feature combinations and model configurations are also evaluated to determine whether a simpler model can achieve similar performance.

Because the model is intended for diabetes prediction, detecting positive diabetes cases is especially important. Therefore, the models are evaluated using accuracy, precision, recall, F1-score, and the number of false positive and false negative predictions. Particular attention is given to reducing false negatives, since failing to identify a person who may have diabetes can be more significant than incorrectly identifying a non-diabetic person as potentially diabetic.

The final goal is to select a model that provides the best balance between predictive performance and model complexity.

# 2. Data understanding

The dataset contains health, lifestyle, and demographic information that can be used to study diabetes. The target variable indicates whether an individual has diabetes or prediabetes.

In [ ]:
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

In [ ]:
# fetch dataset 
cdc_diabetes_health_indicators = fetch_ucirepo(id=891) 

# data (as pandas dataframes) 
X = cdc_diabetes_health_indicators.data.features 
y = cdc_diabetes_health_indicators.data.targets
  
# variable information 
print(cdc_diabetes_health_indicators.variables) 


In [ ]:
X.dtypes

In [ ]:
y.dtypes

In [ ]:
X.head(10)

In [ ]:
import matplotlib.pyplot as plt

def plot_data(X):
    plt.figure(figsize=(20, 15))

    for i, col in enumerate(X.select_dtypes(include=['int', 'float'])):
        plt.subplot(5, 5, i + 1)
        X[col].plot(kind='hist', title=col)
        plt.xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_data(X)

In [ ]:
#correlation matrix
df = pd.concat([X, y], axis=1)

corr = df.corr()

plt.figure(figsize=(10, 10))
plt.imshow(corr, cmap='coolwarm', interpolation='none')
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.show()

## Conclusion

There is multiple colums. All of them are based on query and most of them are binary *`Yes`* / *`No`* based or sex such as *`Diabetes_binary`*, *`HighBP`*, *`HighChol`*, *`CholCheck`*, *`Smoker`*, *`Stroke`*, *`HeartDiseaseorAttack`*, *`PhysActivity`*, *`Fruits`*, *`Veggies`*, *`HvyAlcoholConsump`*, *`AnyHealthcare`*, *`NoDocbcCost`*, *`DiffWalk`* and *`Sex`*. Multiple options were on *`BMI`*, *`GenHlth`*, *`MentHlth`*, *`PhysHlth`*, *`Age`*, *`Education`* and *`Income`*.

Heatmap indicates some correlation between *`Diabetes_binary`* and *`HighBP`*, *`HighChol`*, *`BMI`*, *`HeartDiseaseorAttack`*, *`GenHlth`*, *`PhysHlth`*, *`DiffWalk`* and *`Age`*. The strongest positive correlations appear to be with *`GenHlth`*, *`HighBP`* and *`BMI`*. There are also weaker negative correlations with variables such as *`PhysActivity`*, *`Education`* and *`Income`*.

# 3. Data preparation

First lets make sure there are no empty values.

In [ ]:
X.isna().sum()

In [ ]:
y.isna().sum()

No empty values in data set and no outliers because data is based on questionnaire.

## Logistic regression

The data was split into training and testing sets. The features were also standardized to ensure that they are on a comparable scale. This is useful for logistic regression and makes the model coefficients easier to compare. Shape of y needs to be changed from 2D column vector to 1D vector.

In [ ]:
random_state = 123

### All-model

In [ ]:
y_logistic = y.squeeze()

X_logistic = X

X_train, X_test, y_train, y_test = train_test_split(
    X_logistic,
    y_logistic,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Top5-model

In [ ]:
top5_features = [
    "GenHlth",
    "HighBP",
    "BMI",
    "HighChol",
    "Age"
]

X_top5 = X[top5_features]

X_train_top5, X_test_top5, y_train_top5, y_test_top5 = train_test_split(
    X_top5,
    y_logistic,
    test_size=0.2,
    random_state=random_state,
    stratify=y_logistic
)

scaler_top5 = StandardScaler()

X_train_top5 = scaler_top5.fit_transform(X_train_top5)
X_test_top5 = scaler_top5.transform(X_test_top5)

### Top8-model

In [ ]:
top8_features = [
    "GenHlth",
    "HighBP",
    "BMI",
    "HighChol",
    "Age",
    "Income",
    "DiffWalk",
    "HeartDiseaseorAttack"
]

X_top8 = X[top8_features]

X_train_top8, X_test_top8, y_train_top8, y_test_top8 = train_test_split(
    X_top8,
    y_logistic,
    test_size=0.2,
    random_state=random_state,
    stratify=y_logistic
)

scaler_top8 = StandardScaler()

X_train_top8 = scaler_top8.fit_transform(X_train_top8)
X_test_top8 = scaler_top8.transform(X_test_top8)

### Random Forest - data split

Random Forest does not require feature scaling, so the original features are used directly. The same 80/20 split and random state are used so the results are comparable with the logistic models.

In [ ]:
y_forest = y.squeeze()

X_train_forest, X_test_forest, y_train_forest, y_test_forest = train_test_split(
    X,
    y_forest,
    test_size=0.2,
    random_state=random_state,
    stratify=y_forest
)

# 4. Modeling

A logistic regression model was trained using the training data to predict if an individual has diabetes or prediabetes.

## Logistic

A logistic regression model was trained using the training data to predict if an individual has diabetes or prediabetes. We use three different models based on data set, **`ALL`** with all the data, **`TOP5`** and **`TOP8`** with top permutational importance features. Also all models are tested with **`0.5`** and **`0.35`** as threshold.

### All- model

In [ ]:
logistic_model_all = LogisticRegression(max_iter=1000)

logistic_model_all.fit(X_train, y_train)

y_pred = logistic_model_all.predict(X_test)

### top5_ model

In [ ]:
logistic_model_top5 = LogisticRegression(max_iter=1000)

logistic_model_top5.fit(X_train_top5, y_train_top5)

y_prob_top5 = logistic_model_top5.predict_proba(X_test_top5)[:, 1]
y_pred_top5_035 = (y_prob_top5 >= 0.35).astype(int)

### Top8- model

In [ ]:
logistic_model_top8 = LogisticRegression(max_iter=1000)

logistic_model_top8.fit(X_train_top8, y_train_top8)

y_prob_top8 = logistic_model_top8.predict_proba(X_test_top8)[:, 1]
y_pred_top8_035 = (y_prob_top8 >= 0.35).astype(int)

## Random Forest

Random Forest was chosen as a comparison to logistic regression because it 
can capture non-linear relationships between features and does not require 
feature scaling. The same training and testing split was used so the results 
are directly comparable.

Before training the final model, different values for `max_depth` were 
tested (5, 10, 15, 20, 25).

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score
from sklearn.ensemble import RandomForestClassifier

for depth in [5, 10, 15, 20, 25]:
    model = RandomForestClassifier(
        n_estimators=100,
        bootstrap=True,
        max_depth=depth,
        max_samples=0.75,
        max_features=0.30,
        random_state=random_state
    )

    model.fit(X_train_forest, y_train_forest)

    y_pred = model.predict(X_test_forest)

    print(
        f"Depth {depth}: "
        f"Accuracy={accuracy_score(y_test_forest, y_pred):.4f}, "
        f"Recall={recall_score(y_test_forest, y_pred):.4f}, "
        f"F1={f1_score(y_test_forest, y_pred):.4f}"
    )

### Model selection

Based on the testing results, `max_depth=25` was selected because it provided 
the best recall and F1-score. Accuracy remained nearly constant across all 
depths, so the model that best detects positive diabetes cases was chosen.

In [ ]:

forest_model = RandomForestClassifier(
    n_estimators=100,
    bootstrap=True,
    max_depth=25,
    max_samples=0.75,
    max_features=0.30,
    random_state=random_state
)

forest_model.fit(X_train_forest, y_train_forest)

y_pred_forest = forest_model.predict(X_test_forest)

# 5. Evaluation

Evaluation is made by comparing the performance of the trained models using several classification metrics: accuracy, precision, recall, F1-score, and the numbers of false positive and false negative predictions.

Special attention is given to recall and false negatives because the model is used for diabetes prediction. Missing a person who may have diabetes can be more significant than incorrectly classifying a non-diabetic person as potentially diabetic.

The models are also compared using different classification thresholds to evaluate how the balance between false positives and false negatives changes. The final model is selected based on overall predictive performance and the ability to detect positive diabetes cases.

## Logistic regression model

### All variables model

### Permutation importance evaluatiom

In [ ]:
importance = permutation_importance(
    logistic_model_all,
    X_test,
    y_test,
    n_repeats=10,
    random_state=123,
    scoring="recall"
)

sorted_idx = importance.importances_mean.argsort()

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance.importances_mean
}).sort_values("Importance", ascending=False)

display(importance_df.round(4))

fig, ax = plt.subplots()

ax.boxplot(
    importance.importances[sorted_idx].T,
    vert=False,
    tick_labels=X.columns[sorted_idx]
)

ax.set_title("Permutation Importance")
fig.tight_layout()
plt.show()

### Permutation importance conclusion

Permutation importance indicates that *`GenHlth`* (value: *`0.0886`*), *`HighBP`* (value: *`0.0575`*), *`BMI`* (value: *`0.0557`*), *`HighChol`* (value: *`0.0388`*), *`Age`* (value: *`0.0229`*), *`Income`* (value: *`0.0211`*), *`DiffWalk`* (value: *`0.0148`*) and *`HeartDiseaseorAttack`* (value: *`0.0141`*) show the greatest importance compared to the rest. 

## Evaluating logistic models

Evaluating models by accuracy, precision, recall, F1-score, and the numbers of false positive and false negative predictions, and comparing them with each other.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd


def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }


results = []
results_logistic = []

# Default threshold 0.5
results.append(
    evaluate_model(
        logistic_model_all,
        X_test,
        y_test,
        "All features",
        0.5
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_all,
        X_test,
        y_test,
        "All features",
        0.5
    )
)

results.append(
    evaluate_model(
        logistic_model_top8,
        X_test_top8,
        y_test_top8,
        "Top 8",
        0.5
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_top8,
        X_test_top8,
        y_test_top8,
        "Top 8",
        0.5
    )
)

results.append(
    evaluate_model(
        logistic_model_top5,
        X_test_top5,
        y_test_top5,
        "Top 5",
        0.5
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_top5,
        X_test_top5,
        y_test_top5,
        "Top 5",
        0.5
    )
)



# Threshold 0.35
results.append(
    evaluate_model(
        logistic_model_all,
        X_test,
        y_test,
        "All features",
        0.35
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_all,
        X_test,
        y_test,
        "All features",
        0.35
    )
)

results.append(
    evaluate_model(
        logistic_model_top8,
        X_test_top8,
        y_test_top8,
        "Top 8",
        0.35
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_top8,
        X_test_top8,
        y_test_top8,
        "Top 8",
        0.35
    )
)

results.append(
    evaluate_model(
        logistic_model_top5,
        X_test_top5,
        y_test_top5,
        "Top 5",
        0.35
    )
)

results_logistic.append(
    evaluate_model(
        logistic_model_top5,
        X_test_top5,
        y_test_top5,
        "Top 5",
        0.35
    )
)


comparison_df_logistic = pd.DataFrame(results_logistic)

display(comparison_df_logistic.round(4))

In [ ]:
threshold = 0.35


# All features
y_prob_all = logistic_model_all.predict_proba(X_test)[:, 1]
y_pred_all_035 = (y_prob_all >= threshold).astype(int)

cm_all = confusion_matrix(y_test, y_pred_all_035)

ConfusionMatrixDisplay(
    confusion_matrix=cm_all,
    display_labels=["No diabetes", "Diabetes"]
).plot(cmap="Blues")

plt.title("All Features - Threshold 0.35")
plt.show()


# Top 8
y_prob_top8 = logistic_model_top8.predict_proba(X_test_top8)[:, 1]
y_pred_top8_035 = (y_prob_top8 >= threshold).astype(int)

cm_top8 = confusion_matrix(y_test_top8, y_pred_top8_035)

ConfusionMatrixDisplay(
    confusion_matrix=cm_top8,
    display_labels=["No diabetes", "Diabetes"]
).plot(cmap="Blues")

plt.title("Top 8 Features - Threshold 0.35")
plt.show()


# Top 5
y_prob_top5 = logistic_model_top5.predict_proba(X_test_top5)[:, 1]
y_pred_top5_035 = (y_prob_top5 >= threshold).astype(int)

cm_top5 = confusion_matrix(y_test_top5, y_pred_top5_035)

ConfusionMatrixDisplay(
    confusion_matrix=cm_top5,
    display_labels=["No diabetes", "Diabetes"]
).plot(cmap="Blues")

plt.title("Top 5 Features - Threshold 0.35")
plt.show()

## Random Forest model

The Random Forest model is evaluated with cross-validation, accuracy, precision, recall, F1-score, and a confusion matrix, following the same metrics used for the logistic regression models.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

cv_scores_forest = cross_val_score(
    forest_model,
    X_train_forest,
    y_train_forest,
    cv=10,
    scoring="accuracy"
)

print("Random Forest model:")
print(f"CV accuracy: {cv_scores_forest.mean():.4f}")
print(f"Accuracy: {accuracy_score(y_test_forest, y_pred_forest):.4f}")
print(f"Precision: {precision_score(y_test_forest, y_pred_forest):.4f}")
print(f"Recall: {recall_score(y_test_forest, y_pred_forest):.4f}")
print(f"F1-score: {f1_score(y_test_forest, y_pred_forest):.4f}")

print("")
print(
    classification_report(
        y_test_forest,
        y_pred_forest,
        target_names=["No diabetes", "Diabetes"]
    )
)

cm_forest = confusion_matrix(y_test_forest, y_pred_forest)

ConfusionMatrixDisplay(
    confusion_matrix=cm_forest,
    display_labels=["No diabetes", "Diabetes"]
).plot(cmap="Blues")

plt.title("Random Forest Confusion Matrix")
plt.show()

### Threshold testing

Different thresholds were tested to see how precision, recall, and F1-score 
change. Lower thresholds typically increase recall, which matters here since 
we want to minimize false negatives.

In [ ]:
y_prob_forest = forest_model.predict_proba(X_test_forest)[:, 1]

results_random = []

for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7]:
    row = evaluate_model(
        forest_model,
        X_test_forest,
        y_test_forest,
        "Random Forest",
        threshold
    )
    results.append(row)
    results_random.append(row)

comparison_df_forest = pd.DataFrame(results_random)
display(comparison_df_forest.round(4))

In [ ]:
y_pred_forest_03 = (y_prob_forest >= 0.3).astype(int)

print("Random Forest, threshold 0.3:")
print(f"Accuracy: {accuracy_score(y_test_forest, y_pred_forest_03):.4f}")
print(f"Precision: {precision_score(y_test_forest, y_pred_forest_03):.4f}")
print(f"Recall: {recall_score(y_test_forest, y_pred_forest_03):.4f}")
print(f"F1-score: {f1_score(y_test_forest, y_pred_forest_03):.4f}")

cm_forest_03 = confusion_matrix(y_test_forest, y_pred_forest_03)

ConfusionMatrixDisplay(
    confusion_matrix=cm_forest_03,
    display_labels=["No diabetes", "Diabetes"]
).plot(cmap="Blues")

plt.title("Random Forest Confusion Matrix - Threshold 0.30")
plt.show()

## Conclusions

### Logistic conclusion

The default threshold produces *`952`* false positives and *`6002`* false negatives. This means that relatively few non-diabetic individuals are incorrectly classified as diabetic, but a large number of diabetic individuals are missed by the model.

When the threshold is lowered to *`0.35`*, false negatives decrease from *`6002`* to *`4597`*, while false positives increase from *`952`* to *`2944`*. Therefore, lowering the threshold improves the model's ability to detect diabetes, but at the cost of incorrectly classifying more non-diabetic individuals as diabetic.

In this case, the number of false negatives decreases by *`1405`*, while false positives increase by *`1992`*. For diabetes prediction, reducing false negatives can be more important because missing a person who may have diabetes is generally more serious than incorrectly identifying someone as potentially diabetic.

When comparing the three logistic regression models with a threshold of *`0.35`*, the differences between the models are relatively small. The all-features model achieves an accuracy of *`0.8514`*, precision of *`0.4564`*, recall of *`0.3497`*, and F1-score of *`0.3960`*. The Top 8 model achieves a slightly higher accuracy of *`0.8521`* and precision of *`0.4581`*, but its recall decreases to *`0.3375`* and its F1-score to *`0.3887`*. The Top 5 model also reaches an accuracy of *`0.8521`* with precision of *`0.4578`*, while recall decreases further to *`0.3320`* and the F1-score to *`0.3849`*.

The all-features model therefore performs best overall when the ability to detect diabetic individuals is emphasized. It has the highest recall and F1-score of the three models, meaning that it identifies more positive cases while maintaining the best overall balance between precision and recall.

However, the differences between the models are small. The Top 8 and Top 5 models achieve almost identical accuracy and precision compared with the all-features model, while using fewer input variables. Therefore, although the all-features model is selected as the best-performing model, its advantage over the Top 8 and Top 5 models is relatively minor.

## Random forest conclusions

The Random Forest model outperformed the logistic regression models. The best 
performance was achieved with threshold `0.3`, which gave the highest recall 
(0.4885) and F1-score (0.4511). 

At the default threshold `0.5`, precision (0.5771) and accuracy (0.8660) were 
higher, but recall dropped to 0.1425, resulting in over 6000 missed diabetic 
cases. Lowering the threshold to `0.3` reduced false negatives from 6062 to 
3616, at the cost of more false positives (738 → 4788). This trade-off is 
acceptable because the primary goal is to minimize missed diabetes cases.

Therefore, Random Forest with threshold `0.3` was selected as the final model.

## Logistic regressions models compared to random forests models

In [ ]:
comparison_df = pd.DataFrame(results)

display(comparison_df.round(4))

Tänne teksti yhdistelmä conclusions

# 6. Deployment

These models are recommended to use for evaluating should the patient have further medical examinations such as HbA1c test. The questionnaire suits the best for predicting further exams with **`Random Forest`** with **`0.35`** as threshold as it has lowest false negative count **`4318`**. Second best is **`Logisyic regression ALL -model`** with **`0.35`** threshold as it predicts falsy diabetics **`4597`**.

Decision between **`Random Forest`** and **`Logistic regression`** should be based on computational power where **`Logistic regression`** is less taxing.